<img src="assets/DemoSlidesForExport_0.png" alt="" style="max-width: 100%; max-height: 88vh; width: auto; height: auto; object-fit: contain; display: block; margin: 0 auto;" />

<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star Initialization</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
C-Star consists of python packages that are installable via <code>pip</code> — the standard Python package installer.
</p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Carbon-accounting coastal model simulations and analysis can be run using imported python objects.
</p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Begin by importing python package(s), and checking the environment:
</p>
</div>

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
from IPython.display import Markdown, display

import cstar_forge
import roms_tools as rt

env = cstar_forge.config.get_environment_info()

# Display summary
summary = f"""
### Machine Information
- **Hostname**: `{env.hostname}`
- **System Tag**: `{env.system_tag}`
- **OS**: `{env.os_info}`

### Environment Summary
- **Python Version**: `{env.python_version}`
- **Python Executable**: `{env.python_executable}`
- **Conda/Micromamba Environment**: `{env.env_info}`
- **Kernel**: `{env.kernel_spec}`
"""

display(Markdown(summary))
print(f"C-Star imported: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Develop CDR forcing</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
A CDR intervention can be planned and built by C-Star. Here we add details and define a TracerPerturbation (VolumePerturbation with full tracer control is also available), to be incorporated into C-Star blueprints and ROMS-MARBL inputs in order to add alkalinity as a point souce (gaussian and other non-point-source distrubtions available): </p><br>
</div>

In [ ]:
# make CDR forcing
start_time = datetime(2010, 1, 1)
end_time = datetime(2010, 1, 31)

times = [datetime(2010, 1, 1, 0),
           datetime(2010, 1, 2, 0),
           datetime(2010, 1, 31, 0),
          ]

tracer_fluxes = {"ALK": [0.0, 60.0*10**6, 60.0*10**6]} # meq/s

cdr_tracer_release1 = rt.TracerPerturbation(
name="release_1",
    lat=59.64,  # degree N
    lon=211.12,  # degree E
    depth=2,  # m
    hsc=500,
    vsc=300,
    times=times,
    tracer_fluxes=tracer_fluxes
)

print("CDR options loaded.\n")

<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Define model grid</h1>
    <p style="font-size: large; line-height: 1.45; margin: 0;">
Here we name and define a ROMS model grid using named parameters, grouped into 'settings' and 'boundaries'.
    <p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
The grid and other python objects we established will be used in the next step to establish a CStarSpecBuilder, which will be used to complete C-Star setup.
    </p>
</div>

<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Define the dimensions that will build a ROMS model grid with plain-language names:
</p>

In [ ]:
# Grid parameters ------------------------------------------------------------------
grid_name = "Gulf_of_Alaska_sm"
grid_settings = dict[str, float](
    nx=160,               # X-direction grid cells (xi)
    ny=80,                # Y-direction grid cells (eta)
    size_x=1460,         # km (longitude direction at ~49N)
    size_y=700,         # km (latitude direction)
    center_lon=-150.0,   
    center_lat=57.5,
    rot=41,              # rotation of grid (degrees)
    N=10,                # number of vertical levels
    theta_s=6.0,         # surface control parameter
    theta_b=3.0,         # bottom control parameter
    hc=250.0,            # critical depth
)
boundaries = {
        "south": True,
        "east": False,
        "north": True,
        "west": True, 
}
print("\nGrid options loaded.\n")

<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Generate the full CDR forcing and plot:
</p>

In [ ]:
cdr = rt.CDRForcing(
    grid=rt.Grid(**grid_settings), # for plotting, could omit
    start_time=start_time,
    end_time=end_time,
    releases=[cdr_tracer_release1,],
)

cdr.plot_locations()
#cdr.plot_distribution("my_cdr")
cdr.plot_tracer_flux("ALK")

# Pass the cdr information as a dict, for now:
cdr_as_dict = cdr.model_dump()["CDRForcing"]

<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Create the builder</h1>
    <p style="font-size: large; line-height: 1.45; margin: 0;">
A CstarSpecBuilder instance is created, using a pre-defined ROMS-MARBL model specification (a <b>ModelSpec</b> name, which contains settings to enable ROMS-MARBL features and compilation to match your inputs and environment, and comes from the catalog -- more to come on that), and start and end dates. </p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
The builder takes these changeable options and pulls model <b>source data</b>, generates <b>forcing data</b>, and constructed <b>C-Star blueprints</b> which will run the ROMS model and ensure <b>scientific reproducability<b>:
</p>
</div>

In [ ]:
#Change to roms-tools grid gen, for plotting

# Model setup ------------------------------------------------------------------
model_spec = "cson_roms-marbl_v0.1"

partitioning = {
    "n_procs_x": 2, # number of partitions in xi (x) 
    "n_procs_y": 4, # number of partitions in eta (y) 
}

# Initialize CstarSpecBuilder --------------------------------------------------
builder = cstar_forge.CstarSpecBuilder(
    description="GulfOfAlaska_Demo",
    model_name=model_spec,
    grid_name=grid_name,
    grid_kwargs=grid_settings,
    open_boundaries=boundaries,    
    start_time=start_time,
    end_time=end_time,
    partitioning=partitioning,
    CDR_forcing=cdr_as_dict,
    initialize_catalog_from="local"
    #catalog_root = "local"
)

builder.grid.plot()

<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star Forcing Data and Blueprint Generation</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
C-Star uses online, published datasets for ocean bathymetry, atmospheric variables, and initial and boundary conditions. These datasets are either steamable directly to C-Star or are staged online by C-Worthy. Data sources are specified in the 
</p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Carbon-accounting coastal model simulations and analysis can be run using imported python objects.
</p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Begin by importing python package(s), and checking the environment:
</p>

In [ ]:
#%%prun
#%load_ext line_profiler
# ensure that source data is staged locally
ocn.ensure_source_data()

# prepare model input
#%lprun -f ocn.generate_inputs(clobber=False) # setting clobber=True will overwrite existing files
ocn.generate_inputs(clobber=False) # setting clobber=True will overwrite existing files

# configure and build the model blueprints
ocn.configure_build(
    compile_time_settings={
        # "cdr_output": {
        #    "do_cdr": True
        # }
    }, 
    run_time_settings = {}
    #     "roms.in": {
    #         "time_stepping": {
    #             "dt": 900,
    #         }
    #     }
    # }
)

<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Run ROMS-MARBL from Blueprint</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
Core C-Star functionality, like running a blueprint, is grouped into high-level commands </p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Here we check the blueprint produced in the previous steps, and then compile and ROMS-MARBL and follow the blueprint to perform a model integration.</p>
</div>

In [ ]:
%%time

ocn.prep_cstar_environment(
   account_key = None,  # None gets from machine config or override here
   queue_name = None,  # None gets from machine config or override here
   walltime = "4:00:00",
   clobber = True,  # recommend True, but it will clear previous results from this run
   n_procs_available = 8,  # 0 is auto-detect, change if on a login or shared node to not overuse resources
)

import os
os.environ["THIS_BP_PATH"] = str(ocn.path_blueprint(stage="build"))

!echo $THIS_BP_PATH
!cstar blueprint run $THIS_BP_PATH


<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Analyze ROMS_MARBL output</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
After blueprint or workplan execution finishes, modeled results can be examined and summaried using ROMS-Tools </p><br>
<p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Here we check the blueprint produced in the previous steps, and then compile and ROMS-MARBL and follow the blueprint to perform a model integration.</p>
</div>

In [ ]:
import pandas as pd

cdr_output = rt.ROMSOutput(grid=ocn.grid, path=ocn.run_output_dir / "joined_output" / "output_cdr.*.nc", use_dask=True)

# full output is available as an xarray DataSet as cdr_output.ds
time_index = cdr_output.ds.time.values
print('First, Last output times: ',time_index[0],time_index[-1])

# ROMS-tools plotting is available for results
cdr_output.plot("ALK", time=-1, s=0)
cdr_output.plot("ALK_ALT_CO2", time=-1, s=0)
cdr_output.cdr_metrics()
cdr_output.create_movie(
    "ALK_source",
    time_range=slice(0, -1, 6),                # every 6 output (here, every six hours)
    fps=6,                                     # frames per second
    output_file="simulation_ALK_source.mp4",
    s=0,                                       # surface s=0
    include_boundary=True,                     # include boundary grid cells
    timestamp_xy=(0.05, 0.95)                  # display the time stamp at plot axes (x,y)
)


<div style="background-color: #b8cde0; padding: 30px; min-height: 80vh; width: 100%; box-sizing: border-box;">
<h1 style="margin: 0 0 0.8rem 0; font-size: clamp(1.4rem, 3vw, 2.2rem);">C-Star: Domain Catalog</h1>
<p style="font-size: large; line-height: 1.45; margin: 0;">
Scientifically verified domains and/or models will be available online in a [C]Worthy Domain Catalog. Domain catalogs are groupings of files (DomainSpec, ModelSpec, MachineSpec, and C-Star Blueprints) that together describe repropduceable model experiments. Domain catalogs can be created and hosted a local computer/filesstem, on Github, or elsewhere online.</p>
<br><p style="font-size: large; line-height: 1.45; max-width: 52rem; margin: 0;">
Here we open a DomainCatalog using python, examine the DomainSpec and ModelSpec available, and use them to recreate a simulation.
</p>
</div>

In [ ]:
cat = DomainCatalog(catalog_root=